# 衡量 Chapter 5: Linear Models and Regularization
**Book Reference:** *scikit-learn Cookbook, Third Edition*

---
## 1. Introduction
Basic linear models are often highly susceptible to overfitting by matching the noise in the training data too closely. This chapter explores Regularization techniques (Ridge, Lasso, Elastic Net) to constrain model complexity and improve its ability to generalize to new, unseen data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

%matplotlib inline
np.random.seed(42)

# Simulating a dataset with 50 features, BUT only 10 features actually have an effect (informative)
X, y, true_coef = make_regression(n_samples=200, n_features=50, n_informative=10, 
                                  noise=15, coef=True, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Number of genuinely informative features: {np.sum(true_coef != 0)}")

## 2. Ordinary Least Squares (No Regularization)
Let's see what happens when we use standard linear regression without any penalty constraints.

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

print(f"Linear Regression MSE on Test Data: {mean_squared_error(y_test, lr.predict(X_test)):.2f}")
print(f"Number of features used by the model (coefficient != 0): {np.sum(lr.coef_ != 0)}")

## 3. Ridge Regression (L2 Regularization)
Ridge Regression shrinks all coefficients to prevent the model from becoming overly dependent on a handful of features. We use `RidgeCV` to automatically discover the optimal `alpha` (penalty strength) value.

In [ ]:
from sklearn.linear_model import RidgeCV

# Search for the best alpha among 0.1, 1.0, 10.0, and 100.0
ridge = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5)
ridge.fit(X_train, y_train)

print(f"Best alpha found by RidgeCV: {ridge.alpha_}")
print(f"Ridge Regression MSE on Test Data: {mean_squared_error(y_test, ridge.predict(X_test)):.2f}")
print(f"Number of features used by the model (coefficient != 0): {np.sum(ridge.coef_ != 0)}")

# Note: Ridge never drops coefficients exactly to zero, it only makes them very small.

## 4. Lasso Regression (L1 Regularization)
Lasso (Least Absolute Shrinkage and Selection Operator) is exceptionally effective for datasets containing substantial noise (like this one). Lasso suppresses uninformative feature coefficients until they are **exactly zero**.

In [ ]:
from sklearn.linear_model import LassoCV

lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train, y_train)

print(f"Best alpha found by LassoCV: {lasso.alpha_:.4f}")
print(f"Lasso Regression MSE on Test Data: {mean_squared_error(y_test, lasso.predict(X_test)):.2f}")
print(f"Number of features used by the model (coefficient != 0): {np.sum(lasso.coef_ != 0)}")
print("Lasso successfully discarded most noise and retained only relevant features!")

## 5. Visualizing Coefficient Comparisons
Let's compare the coefficient values across standard Linear Regression, Ridge, and Lasso models visually.

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(lr.coef_, 'o', label='Linear Regression', alpha=0.5, markersize=8)
plt.plot(ridge.coef_, '^', label='Ridge', alpha=0.7, markersize=8)
plt.plot(lasso.coef_, 's', label='Lasso', alpha=0.7, markersize=8)
plt.axhline(0, color='black', linestyle='--')

plt.xlabel('Feature Index (0 - 49)')
plt.ylabel('Coefficient Value')
plt.title('Coefficient Comparison: Standard Linear Regression vs Ridge vs Lasso')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## 6. Elastic Net
Elastic Net combines the L1 (Lasso) and L2 (Ridge) penalty properties. This is highly useful when managing multi-collinear variables; where Lasso randomly preserves a single feature from a group of highly correlated predictors, Elastic Net tends to shrink and retain the entire group together.

In [ ]:
from sklearn.linear_model import ElasticNetCV

# l1_ratio represents the ratio of L1 penalty. Setting l1_ratio=1 equivalent to 100% Lasso.
elastic = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 1.0], cv=5, random_state=42)
elastic.fit(X_train, y_train)

print(f"Best l1_ratio: {elastic.l1_ratio_}")
print(f"Best alpha: {elastic.alpha_:.4f}")
print(f"Elastic Net MSE on Test Data: {mean_squared_error(y_test, elastic.predict(X_test)):.2f}")
print(f"Number of features used: {np.sum(elastic.coef_ != 0)}")